# 🚀 Titanic Survival Prediction: Modeling

This notebook implements the preprocessing pipeline and trains a baseline XGBoost model to predict passenger survival.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve, auc, confusion_matrix, classification_report
from xgboost import XGBClassifier
import sys
import os

# Add src to path so we can import our preprocessing pipeline
sys.path.append(os.path.abspath('../'))
from src.preprocessing import PreprocessingPipeline

print("Imports complete.")

## 1. Data Loading

In [ ]:
from src.data_loader import load_train, load_test

train_df = load_train()
test_df = load_test()

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

## 2. Preprocessing

In [ ]:
# Preprocessing pipeline instance
# We will fit this inside the CV loop to prevent data leakage
pipeline = PreprocessingPipeline()

print("Preprocessing pipeline initialized. Fitting will occur inside the CV loop to prevent data leakage.")

## 3. Validation Split

In [ ]:
# Target variable from raw training data
y = train_df['Survived']

print(f"Target shape: {y.shape}")

## 4. Baseline XGBoost Model

In [ ]:
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

# Final optimal hyperparameters
best_params = {
    'max_depth': 3,
    'learning_rate': 0.01,
    'subsample': 0.9,
    'colsample_bytree': 0.8,
    'scale_pos_weight': 1
}

gkf = GroupKFold(n_splits=5)

oof_probs = np.zeros(len(train_df))
cv_scores = []
best_iterations = []

print(f"Evaluating model with optimal params: {best_params} using GroupKFold")

# Leak-free CV evaluation grouped by Ticket
for train_idx, val_idx in gkf.split(train_df, y, groups=train_df['Ticket']):
    # Split raw data
    train_fold_raw = train_df.iloc[train_idx]
    val_fold_raw = train_df.iloc[val_idx]
    
    # Fit and transform training fold to record feature columns
    fold_pipeline = PreprocessingPipeline()
    X_train_cv_full = fold_pipeline.fit_transform(train_fold_raw)
    X_train_cv = X_train_cv_full.drop(columns=['Survived', 'PassengerId'], errors='ignore')
    
    # Transform validation fold using recorded feature columns
    X_val_cv_full = fold_pipeline.transform(val_fold_raw)
    X_val_cv = X_val_cv_full.drop(columns=['Survived', 'PassengerId'], errors='ignore')
    
    y_train_cv = train_fold_raw['Survived']
    y_val_cv = val_fold_raw['Survived']
    
    model = XGBClassifier(
        max_depth=best_params['max_depth'],
        learning_rate=best_params['learning_rate'],
        subsample=best_params['subsample'],
        colsample_bytree=best_params['colsample_bytree'],
        scale_pos_weight=best_params['scale_pos_weight'],
        n_estimators=1000,
        random_state=42,
        early_stopping_rounds=50,
        eval_metric='logloss'
    )
    
    model.fit(
        X_train_cv, y_train_cv,
        eval_set=[(X_val_cv, y_val_cv)],
        verbose=False
    )
    
    val_probs = model.predict_proba(X_val_cv)[:, 1]
    oof_probs[val_idx] = val_probs
    cv_scores.append(accuracy_score(y_val_cv, (val_probs >= 0.5).astype(int)))
    best_iterations.append(model.best_iteration)

# Update global variables for later cells
oof_probs_best = oof_probs
best_score = np.mean(cv_scores)
best_n_estimators = int(np.mean(best_iterations))

print(f"Leak-Free GroupCV Accuracy: {best_score:.4f}")
print(f"Optimal n_estimators (avg): {best_n_estimators}")

## 5. Evaluation

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, roc_curve, auc, confusion_matrix

# Find decision threshold maximizing OOF Accuracy
best_threshold = 0.5
max_acc = 0.0

# Reset range to 0.48 - 0.50 to reduce false positive noise
for threshold in np.arange(0.48, 0.51, 0.01):
    y_pred_thresh = (oof_probs_best >= threshold).astype(int)
    acc = accuracy_score(y, y_pred_thresh)
    if acc > max_acc:
        max_acc = acc
        best_threshold = threshold

print(f"Optimal Threshold: {best_threshold:.2f}")
print(f"Max OOF Accuracy: {max_acc:.4f}")

# Final OOF evaluation
y_pred_final = (oof_probs_best >= best_threshold).astype(int)
print("\nFinal OOF Classification Report:\n", classification_report(y, y_pred_final))

# Visualizations: Confusion Matrix and ROC Curve
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

sns.heatmap(confusion_matrix(y, y_pred_final), annot=True, fmt='d', cmap='Blues', ax=ax1)
ax1.set_title(f'OOF Confusion Matrix (Thresh = {best_threshold:.2f})')
ax1.set_xlabel('Predicted')
ax1.set_ylabel('Actual')

fpr, tpr, _ = roc_curve(y, oof_probs_best)
roc_auc = auc(fpr, tpr)
ax2.plot(fpr, tpr, color='darkorange', lw=2, label=f'OOF ROC curve (AUC = {roc_auc:.2f})')
ax2.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
ax2.set_xlim([0.0, 1.0])
ax2.set_ylim([0.0, 1.05])
ax2.set_xlabel('False Positive Rate')
ax2.set_ylabel('True Positive Rate')
ax2.set_title('OOF ROC Curve')
ax2.legend(loc="lower right")

plt.tight_layout()
plt.show()

## 6. Final Prediction and Submission

In [ ]:
import os
import pandas as pd

# 1. Final Preprocessing: Fit pipeline on 100% of training data
final_pipeline = PreprocessingPipeline()
final_pipeline.fit(train_df)

X_full = final_pipeline.transform(train_df).drop(columns=['Survived', 'PassengerId'], errors='ignore')
y_full = train_df['Survived']

# 2. Final Model Training: Use optimal hyperparameters
final_model = XGBClassifier(
    max_depth=best_params['max_depth'],
    subsample=best_params['subsample'],
    learning_rate=best_params['learning_rate'],
    colsample_bytree=0.8,
    n_estimators=best_n_estimators,
    random_state=42,
    eval_metric='logloss'
)
final_model.fit(X_full, y_full)

# 3. Test Set Prediction: Transform test data using final_pipeline
X_test = final_pipeline.transform(test_df).drop(columns=['Survived', 'PassengerId'], errors='ignore')
test_probs = final_model.predict_proba(X_test)[:, 1]

# 4. Apply optimal threshold
test_preds = (test_probs >= best_threshold).astype(int)

# Create submission file
submission = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Survived': test_preds
})

os.makedirs('../outputs', exist_ok=True)
submission.to_csv('../outputs/submission.csv', index=False)
print("Submission file saved to outputs/submission.csv")